In [1]:
import torch
from torch.utils.data import Dataset
import cv2 as cv
import os
import pickle
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.models import ResNet50_Weights
from torch.utils.data import random_split
from torchvision.transforms import Compose, Normalize, Resize, ToTensor
from tqdm import tqdm

In [2]:
# Tải file nén chứa toàn bộ ảnh (.tgz) từ server của Fast.ai
!wget https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz

# Giải nén file vừa tải
!tar -xzf cifar10.tgz

--2026-03-22 04:17:03--  https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.217.142.16, 16.15.237.0, 16.182.41.160, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.217.142.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 135107811 (129M) [application/x-tar]
Saving to: ‘cifar10.tgz’

cifar10.tgz         100%[===================>] 128.85M  47.1MB/s    in 2.7s    

2026-03-22 04:17:07 (47.1 MB/s) - ‘cifar10.tgz’ saved [135107811/135107811]



In [3]:
class dataset(Dataset):
  def __init__(self, path, compose = None):
    super().__init__()
    self.path_folder = path
    self.compose = compose
    self.labels = []
    self.imgs = []

    labels = sorted(os.listdir(path))
    encode_labels = {}
    for i, name in enumerate(labels):
      encode_labels[name] = i

    for i in labels:
      imgs = os.listdir(os.path.join(self.path_folder, i))
      for j in imgs:
        self.imgs.append(os.path.join(self.path_folder, i, j))
        self.labels.append(encode_labels[i])

  def __getitem__(self, index):
    img = cv.imread(self.imgs[index])
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

    label = self.labels[index]

    if self.compose is not None:
      img = self.compose(img)


    return img, label

  def __len__(self):
    return len(self.labels)


### https://www.digitalocean.com/community/tutorials/popular-deep-learning-architectures-resnet-inceptionv3-squeezenet

(imgz + 2*Padding -kernel + 1)/2  

In [4]:
class BotNeck(nn.Module):
  def __init__(self,input_channels, output_channels, stride):
    super().__init__()

    botneck = output_channels//4
    self.conv = nn.Sequential(
        nn.Conv2d(in_channels= input_channels, out_channels= botneck, kernel_size=1, padding=0, stride=1, bias= False),
        nn.BatchNorm2d(botneck),
        nn.ReLU(),
        nn.Conv2d(in_channels= botneck, out_channels= botneck, kernel_size=3, padding=1, stride=stride, bias= False),
        nn.BatchNorm2d(botneck),
        nn.ReLU(),
        nn.Conv2d(in_channels= botneck, out_channels= output_channels, kernel_size=1, padding=0, stride=1, bias= False),
        nn.BatchNorm2d(output_channels),
    )

    self.shortcut = nn.Identity()

    if stride != 1 or input_channels != output_channels:
      self.shortcut = nn.Sequential(
          nn.Conv2d(input_channels, output_channels, kernel_size=1, stride=stride, bias= False),
          nn.BatchNorm2d(output_channels)
      )
    self.relu = nn.ReLU()

  def forward(self, x):
    x_shortcut = self.shortcut(x)
    x = self.conv(x)
    x = x + x_shortcut
    x = self.relu(x)
    return x


In [5]:
class Layer(nn.Module):
  def __init__(self, in_channels, out_channels, stride, number):
    super().__init__()
    layer=[]
    layer.append(BotNeck(in_channels, out_channels, stride))

    for i in range(number-1):
      layer.append(BotNeck(out_channels, out_channels, 1))

    self.layer = nn.Sequential(*layer)
  def forward(self,x):
    x = self.layer(x)
    return x

In [6]:
class ResNet50(nn.Module):
  def __init__(self, numbers =[3,4,6,3], nums_class = 10):
    super().__init__()
    self.conv1 = nn.Sequential(
        nn.Conv2d(3, 64, kernel_size=7, padding=3, stride=2, bias= False),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, padding=1, stride=2)
    )

    in_channels = [64, 256, 512, 1024]
    out_channels = [256, 512, 1024, 2048]
    strides = [1, 2, 2, 2]
    all_layers = []
    for i in range(len(numbers)):
      all_layers.append(Layer(in_channels[i], out_channels[i],strides[i], numbers[i]))
    self.all_layers = nn.Sequential(*all_layers)

    self.avgpool = nn.AdaptiveAvgPool2d((1,1))
    self.flatten = nn.Flatten()
    self.fc = nn.Linear(2048, nums_class)
  def forward(self, x):
    x = self.conv1(x)
    x = self.all_layers(x)
    x = self.avgpool(x)
    x = self.flatten(x)
    x = self.fc(x)
    return x



In [7]:
model = ResNet50()

weights = ResNet50_Weights.IMAGENET1K_V2

# 2. Lấy state_dict (Nó sẽ tự tải về nếu máy bạn chưa có)
official_state_dict = weights.get_state_dict(check_hash=True)
print("Tên lớp của tôi:", model.state_dict().keys())
print("Tên lớp của Torch:", official_state_dict.keys())

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 120MB/s]

Tên lớp của tôi: odict_keys(['conv1.0.weight', 'conv1.1.weight', 'conv1.1.bias', 'conv1.1.running_mean', 'conv1.1.running_var', 'conv1.1.num_batches_tracked', 'all_layers.0.layer.0.conv.0.weight', 'all_layers.0.layer.0.conv.1.weight', 'all_layers.0.layer.0.conv.1.bias', 'all_layers.0.layer.0.conv.1.running_mean', 'all_layers.0.layer.0.conv.1.running_var', 'all_layers.0.layer.0.conv.1.num_batches_tracked', 'all_layers.0.layer.0.conv.3.weight', 'all_layers.0.layer.0.conv.4.weight', 'all_layers.0.layer.0.conv.4.bias', 'all_layers.0.layer.0.conv.4.running_mean', 'all_layers.0.layer.0.conv.4.running_var', 'all_layers.0.layer.0.conv.4.num_batches_tracked', 'all_layers.0.layer.0.conv.6.weight', 'all_layers.0.layer.0.conv.7.weight', 'all_layers.0.layer.0.conv.7.bias', 'all_layers.0.layer.0.conv.7.running_mean', 'all_layers.0.layer.0.conv.7.running_var', 'all_layers.0.layer.0.conv.7.num_batches_tracked', 'all_layers.0.layer.0.shortcut.0.weight', 'all_layers.0.layer.0.shortcut.1.weight', 'all_la

In [8]:
def load_model():

  model = ResNet50()

  mydict= model.state_dict()
  office= weights.get_state_dict(check_hash=True)

  # Khác class numbers (1000 và 10)
  office.pop('fc.weight', None)
  office.pop('fc.bias', None)

  list_key_r50= list(office.keys())
  list_key_mymodel = list(mydict.keys())

  new_dict ={}
  for i in range(len(list_key_r50)):
    if office[list_key_r50[i]].shape != mydict[list_key_mymodel[i]].shape:
      print(f"shape khác nhau ở: {list_key_r50[i]} và {list_key_mymodel[i]}")

    else:
      new_dict[list_key_mymodel[i]] = office[list_key_r50[i]]

  # Strict để bỏ qua các layer bị thiếu
  model.load_state_dict(new_dict, strict=False)
  # print(f"Xong: {model.state_dict()}")
  return model

# Train

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [14]:

def train(resume = None, epochs = 3):
  device = "cuda" if torch.cuda.is_available() else "cpu"

  compose = Compose([
      ToTensor(),
      Resize((224,224)),
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
  ])

  training_data = dataset("/content/cifar10/train", compose)

  generator = torch.Generator().manual_seed(42)
  train_ds, val_ds = random_split(training_data, [0.9, 0.1], generator=generator)

  train_dataloader = DataLoader(train_ds, batch_size=32, shuffle=True)
  val_dataloader = DataLoader(val_ds, batch_size=32, shuffle=False)

  model = load_model().to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
  criterion = nn.CrossEntropyLoss()

  if resume is not None:
    sta_dict = torch.load(resume, map_location=device) if isinstance(resume, str) else resume

    epoch = sta_dict['epoch']
    model.load_state_dict(sta_dict['last_model'])
    optimizer.load_state_dict(sta_dict['optimizer'])
    best_loss = sta_dict['best_loss']
    best_model = sta_dict['best_model']

  else:
    sta_dict = {}
    epoch = 0
    best_loss = 10000000


  for epoch in range(epoch, epochs):
    print(f"Epoch: {epoch+1}")
    total_loss = 0

    progress = tqdm(train_dataloader, desc=f"Train Epoch {epoch+1}")
    model.train()
    for idx, (imgs, labels) in (enumerate(progress)):
      imgs = imgs.to(device)
      labels = labels.to(device)

      outputs= model(imgs)
      loss = criterion(outputs, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      total_loss += loss.item()
      progress.set_postfix(loss=loss.item())
    print(f"Loss train: {total_loss/len(train_dataloader)}")

    model.eval()
    total_loss = 0
    all_labels = []
    all_predict = []
    with torch.no_grad():
      for idx, (imgs, labels) in enumerate(val_dataloader):
        imgs = imgs.to(device)
        labels = labels.to(device)

        all_labels.extend(labels.cpu().tolist())
        outputs= model(imgs)
        all_predict.extend(torch.argmax(outputs, dim=1).cpu().tolist())
        loss = criterion(outputs, labels)

        total_loss += loss.item()

    print(f"Loss val: {total_loss/len(val_dataloader)}")
    print(accuracy_score(all_labels, all_predict))

    if total_loss/len(val_dataloader) < best_loss:
      best_loss = total_loss/len(val_dataloader)
      sta_dict["best_loss"] = total_loss/len(val_dataloader)
      sta_dict["best_model"] = model.state_dict()
      sta_dict['best_opimizer'] = optimizer.state_dict()
      sta_dict['best_epoch'] = epoch
      torch.save(sta_dict, "best_model.pth")
      print("Save best model")

    sta_dict["last_model"] = model.state_dict()
    sta_dict['optimizer'] = optimizer.state_dict()
    sta_dict['epoch'] = epoch
    torch.save(sta_dict, "last_model.pth")
    print("Save last model")






In [15]:
train()

Epoch: 1


Train Epoch 1: 100%|██████████| 1407/1407 [09:08<00:00,  2.56it/s, loss=0.179]


Loss train: 0.3448009351976666
Loss val: 0.13977806239869375
0.9524
Save best model
Save last model
Epoch: 2


Train Epoch 2: 100%|██████████| 1407/1407 [09:06<00:00,  2.57it/s, loss=0.0398]


Loss train: 0.11388064787955458
Loss val: 0.1497428530032278
0.9542
Save last model
Epoch: 3


Train Epoch 3: 100%|██████████| 1407/1407 [09:06<00:00,  2.57it/s, loss=0.332]


Loss train: 0.07141036291280066
Loss val: 0.14233199679767297
0.9596
Save last model


In [18]:
compose = Compose([
    ToTensor(),
    Resize((224,224)),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_data = dataset("/content/cifar10/test", compose)
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=False)

total_loss = 0
progress = tqdm(test_dataloader)
device = "cuda" if torch.cuda.is_available() else "cpu"

path_best = "/content/best_model.pth"
model = ResNet50().to(device)
sta_dict = torch.load(path_best, map_location=device)

model.load_state_dict(sta_dict['best_model'])

predict = []
true_labels = []

model.eval()
with torch.no_grad():
  for idx, (imgs, labels) in enumerate(progress):
    imgs = imgs.to(device)
    labels = labels.to(device)

    outputs= model(imgs)
    predict.extend(torch.argmax(outputs, dim=1).cpu().tolist())
    true_labels.extend(labels.cpu().tolist())


print(accuracy_score(true_labels, predict))
print(classification_report(true_labels, predict))
print(confusion_matrix(true_labels, predict))






100%|██████████| 313/313 [00:46<00:00,  6.73it/s]

0.9517
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      1000
           1       0.97      0.98      0.98      1000
           2       0.92      0.96      0.94      1000
           3       0.91      0.90      0.90      1000
           4       0.97      0.95      0.96      1000
           5       0.93      0.90      0.92      1000
           6       0.98      0.97      0.97      1000
           7       0.97      0.96      0.97      1000
           8       0.95      0.99      0.97      1000
           9       0.99      0.96      0.97      1000

    accuracy                           0.95     10000
   macro avg       0.95      0.95      0.95     10000
weighted avg       0.95      0.95      0.95     10000

[[948   1  22   2   0   0   1   0  21   5]
 [  2 985   0   0   0   0   0   0   5   8]
 [  7   0 958   9   9   3  10   2   2   0]
 [  5   1  20 895   7  55   4   3  10   0]
 [  4   0  10  13 953   4   5  10   1   0]
 [  2   0  16  56  